# Reusable template — incident / case-priority classification

**Short name (GitHub):** `CrimRisk` template.

Swap CSV, target, missing column, costly class. Trees first. Never add protected-class fields to “improve” the score.

Not bail. Not sentencing. Not profiling.



## Knobs


In [ ]:
DATA_PATH = "data/incidents.csv"
TARGET = "priority"
POSITIVE_LABEL = "p"    # costly miss when this is predicted as the other class
MISSING_COL = "clearance"
MISSING_TOKEN = "?"
MISSING_FILL = "u"
DROP_CONST = True
TEST_SIZE = 0.20
SEED = 42
N_ESTIMATORS = 100


## Pipeline


In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

df = pd.read_csv(DATA_PATH)
print(df.shape, df[TARGET].value_counts().to_dict())
df[MISSING_COL] = df[MISSING_COL].replace(MISSING_TOKEN, MISSING_FILL)
df = df.drop_duplicates().reset_index(drop=True)
if DROP_CONST:
    const = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
    print("drop", const)
    df = df.drop(columns=const)
work = df.copy()
encoders = {}
for c in work.columns:
    le = LabelEncoder(); work[c] = le.fit_transform(work[c].astype(str)); encoders[c] = le
print("class map", dict(zip(encoders[TARGET].classes_, encoders[TARGET].transform(encoders[TARGET].classes_))))
X = work.drop(columns=[TARGET]); y = work[TARGET]
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
clf = RandomForestClassifier(n_estimators=N_ESTIMATORS, random_state=SEED)
clf.fit(Xtr, ytr)
yp = clf.predict(Xte)
print("acc", accuracy_score(yte, yp))
print(confusion_matrix(yte, yp))
print(classification_report(yte, yp))

